In [ ]:
from private_config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#主要变化：使用SiLU取代ReLU,优化模型架构，以此作为新模型迭代起点
from pathlib import Path
import json
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
from data_loader_clean import *
from GPN_V4_adaMulti_test import *
import higher
# from draw_pic import *
from thop import profile, clever_format
from para_and_indexes_collection_v1 import FewShotDeploymentProfiler
#约定：所有训练时使用类数<configured>抽<configured>，总样本<configured>个，每个任务中样本数支持集<configured>，查询集<configured>，
#     所有测试时使用类数<configured>抽<configured>-<configured>，总样本<configured>个，每个任务中样本数支持集<configured>,<configured>,<configured>，查询集<configured>，

# 参数设置
N_WAY = _cfg_require('GPN_V3_clean.ipynb.cell0.module.N_WAY')
K_SHOT = _cfg_require('GPN_V3_clean.ipynb.cell0.module.K_SHOT')
Q_QUERY = _cfg_require('GPN_V3_clean.ipynb.cell0.module.Q_QUERY')
NUM_TASKS = _cfg_require('GPN_V3_clean.ipynb.cell0.module.NUM_TASKS')
EPOCHS = _cfg_require('GPN_V3_clean.ipynb.cell0.module.EPOCHS')
BLR = _cfg_require('GPN_V3_clean.ipynb.cell0.module.BLR')
BATCH_SIZE =_cfg_require('GPN_V3_clean.ipynb.cell0.module.BATCH_SIZE')
SEED = _cfg_require('GPN_V3_clean.ipynb.cell0.module.SEED')#<configured>
SAVE_MODEL_DIR = Path(_cfg_require('GPN_V3_clean.ipynb.cell0.module.path'))
SAVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
SAVE_MODEL_PATH=SAVE_MODEL_DIR/_cfg_require('GPN_V3_clean.ipynb.cell0.module.path__2')
LOAD_MODEL_PATH=_cfg_require('GPN_V3_clean.ipynb.cell0.module.LOAD_MODEL_PATH')

# Historical configuration omitted; provide values through private configuration.
dataset_path = _cfg_require('GPN_V3_clean.ipynb.cell0.module.dataset_path')


def main():
    
    train_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=True,train_ratio=_cfg_require('GPN_V3_clean.ipynb.cell0.main.train_ratio'),max_sample_count=_cfg_require('GPN_V3_clean.ipynb.cell0.main.max_sample_count'),
                            )
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,train_ratio=_cfg_require('GPN_V3_clean.ipynb.cell0.main.train_ratio__2'),max_sample_count=_cfg_require('GPN_V3_clean.ipynb.cell0.main.max_sample_count__2'),
                            )

    meta_train_dataset = MetaDataset(train_dataset, 8,15,5,num_tasks=NUM_TASKS)
    meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,num_tasks=_cfg_require('GPN_V3_clean.ipynb.cell0.main.num_tasks'))
    
    # DataLoader
    train_loader = DataLoader(meta_train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=_cfg_require('GPN_V3_clean.ipynb.cell0.main.num_workers'))
    test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('GPN_V3_clean.ipynb.cell0.main.batch_size'), shuffle=True)

    #创建模型
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = GPN_Optimized(
                 reduction=_cfg_require('GPN_V3_clean.ipynb.cell0.main.reduction'))

    params = count_parameters(model)
    
    
    x=torch.randn(2,1,_cfg_require('GPN_V3_clean.ipynb.cell0.main.size_or_budget'),_cfg_require('GPN_V3_clean.ipynb.cell0.main.size_or_budget__2'))
    # 前向传播
    v, s = model(x)
    print(f"Embedding features (v) shape: {v.shape}")
    print(f"Precision features (s) shape: {s.shape}")
    print(f"Precision values are positive: {(s > 0).all().item()}")

    # 带中间特征输出
    v, s, features = model(x, return_intermediate=True)
    print(f"\nIntermediate features:")
    for name, feat in features.items():
        if isinstance(feat, torch.Tensor):
            print(f"  {name}: {feat.shape}")
    input_tensor = torch.randn(1, 1, _cfg_require('GPN_V3_clean.ipynb.cell0.main.size_or_budget__3'), _cfg_require('GPN_V3_clean.ipynb.cell0.main.size_or_budget__4'))
    
    flops, params = profile(model, inputs=(input_tensor,))
    flops, params = clever_format([flops, params], "%.3f")
    print(f"FLOPs: {flops}, Params: {params}")
    # 创建训练器
    meta_learner = GPNTrainer(model,device=device,
                              mlr=BLR,use_Mdistance=_cfg_require('GPN_V3_clean.ipynb.cell0.main.use_Mdistance'),use_multi=_cfg_require('GPN_V3_clean.ipynb.cell0.main.use_multi'),save_middle=_cfg_require('GPN_V3_clean.ipynb.cell0.main.save_middle'),
                              crop_ratio_h=_cfg_require('GPN_V3_clean.ipynb.cell0.main.crop_ratio_h'),crop_ratio_l=_cfg_require('GPN_V3_clean.ipynb.cell0.main.crop_ratio_l'),resize=_cfg_require('GPN_V3_clean.ipynb.cell0.main.resize'),
                              merge_threshold=_cfg_require('GPN_V3_clean.ipynb.cell0.main.merge_threshold'),merge_method=_cfg_require('GPN_V3_clean.ipynb.cell0.main.merge_method'))#('connected'连通分量'pairwise'成对合并 'hierarchical'层次聚类'dbscan'基于密度的聚类方法)
    print("开始训练...")
    meta_learner.train(train_loader=train_loader,test_loader=test_loader,epochs=EPOCHS,n_ways=N_WAY,save_path=SAVE_MODEL_PATH)
    print("训练完成！")
    
def inference():
    '\n    推理示例：加载已保存的模型进行预测\n    \n    !!!若报模型结构错误，检查：<configured>.引用是否为对应模型\n    '
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 创建模型
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,
                            train_ratio=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.train_ratio'),max_sample_count=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.max_sample_count'),
                            )
    model = GPN_Optimized(
                 reduction=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.reduction'),
                 )
    
    meta_learner = GPNTrainer(model,device=device,
                              mlr=BLR,use_Mdistance=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.use_Mdistance'),use_multi=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.use_multi'),save_middle=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.save_middle'),
                              crop_ratio_h=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.crop_ratio_h'),crop_ratio_l=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.crop_ratio_l'),resize=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.resize'),
                              merge_threshold=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.merge_threshold'),merge_method=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.merge_method'))
    meta_learner.load_model(LOAD_MODEL_PATH)
    
    profiler = FewShotDeploymentProfiler(
        trainer=meta_learner,
        device=device,
        input_size=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.input_size'),
    )

    report_amgpn = profiler.generate_profile_report(
        test_dataset=test_dataset,
        n_way=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.n_way'),
        k_shot=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.k_shot'),
        q_query=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.q_query'),
        n_warmup=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.n_warmup'),
        n_trials=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.n_trials'),
        num_workers=_cfg_require('GPN_V3_clean.ipynb.cell0.inference.num_workers'),
    )
    PROFILE_DIR = Path(_cfg_require('GPN_V3_clean.ipynb.cell0.inference.path'))
    PROFILE_DIR.mkdir(parents=True, exist_ok=True)

    # 保存 JSON，保留完整字段
    with open(PROFILE_DIR / "report_amgpn.json", "w", encoding="utf-8") as f:
        json.dump(report_amgpn, f, indent=4, ensure_ascii=False)

    # 保存 CSV，方便论文表格整理
    pd.DataFrame([report_amgpn]).to_csv(
        PROFILE_DIR / "report_amgpn.csv",
        index=False,
        encoding="utf-8-sig"
    )
    # 评估
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.

    # meta_learner.plot_task_proto_assignment_heatmap(
    #     test_dataset,
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    #     sort_queries=True,
    #     show_values=False
    # )
    
    print("推理完成")
if __name__ == '__main__':
    # main()
    inference()
    
# % 时间计算：单任务构建+预测时间
# Historical configuration omitted; provide values through private configuration.
# % hierarchical：<configured>3ms
# % pairwise：<configured>5ms
# % dbscan：<configured>9ms
# % IMP：<configured>8ms
# %均采用warm-up策略去除第一个任务

In [ ]:
from private_config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#主要变化：使用SiLU取代ReLU,优化模型架构，以此作为新模型迭代起点
from pathlib import Path
import json
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
from data_loader_clean import *
from GPN_V4_adaMulti_test import *
import higher
# from draw_pic import *
from thop import profile, clever_format
from para_and_indexes_collection_v1 import FewShotDeploymentProfiler
#约定：所有训练时使用类数<configured>抽<configured>，总样本<configured>个，每个任务中样本数支持集<configured>，查询集<configured>，
#     所有测试时使用类数<configured>抽<configured>-<configured>，总样本<configured>个，每个任务中样本数支持集<configured>,<configured>,<configured>，查询集<configured>，

# 参数设置
N_WAY = _cfg_require('GPN_V3_clean.ipynb.cell1.module.N_WAY')
K_SHOT = _cfg_require('GPN_V3_clean.ipynb.cell1.module.K_SHOT')
Q_QUERY = _cfg_require('GPN_V3_clean.ipynb.cell1.module.Q_QUERY')
NUM_TASKS = _cfg_require('GPN_V3_clean.ipynb.cell1.module.NUM_TASKS')
EPOCHS = _cfg_require('GPN_V3_clean.ipynb.cell1.module.EPOCHS')
BLR = _cfg_require('GPN_V3_clean.ipynb.cell1.module.BLR')
BATCH_SIZE =_cfg_require('GPN_V3_clean.ipynb.cell1.module.BATCH_SIZE')
SEED = _cfg_require('GPN_V3_clean.ipynb.cell1.module.SEED')#<configured>
SAVE_MODEL_DIR = Path(_cfg_require('GPN_V3_clean.ipynb.cell1.module.path'))
SAVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
SAVE_MODEL_PATH=SAVE_MODEL_DIR/_cfg_require('GPN_V3_clean.ipynb.cell1.module.path__2')
LOAD_MODEL_PATH=_cfg_require('GPN_V3_clean.ipynb.cell1.module.LOAD_MODEL_PATH')

# Historical configuration omitted; provide values through private configuration.
dataset_path = _cfg_require('GPN_V3_clean.ipynb.cell1.module.dataset_path')


def main():
    
    train_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=True,train_ratio=_cfg_require('GPN_V3_clean.ipynb.cell1.main.train_ratio'),max_sample_count=_cfg_require('GPN_V3_clean.ipynb.cell1.main.max_sample_count'),
                            )
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,train_ratio=_cfg_require('GPN_V3_clean.ipynb.cell1.main.train_ratio__2'),max_sample_count=_cfg_require('GPN_V3_clean.ipynb.cell1.main.max_sample_count__2'),
                            )

    meta_train_dataset = MetaDataset(train_dataset, 8,15,5,num_tasks=NUM_TASKS)
    meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,num_tasks=_cfg_require('GPN_V3_clean.ipynb.cell1.main.num_tasks'))
    
    # DataLoader
    train_loader = DataLoader(meta_train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=_cfg_require('GPN_V3_clean.ipynb.cell1.main.num_workers'))
    test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('GPN_V3_clean.ipynb.cell1.main.batch_size'), shuffle=True)

    #创建模型
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = GPN_Optimized(
                 reduction=_cfg_require('GPN_V3_clean.ipynb.cell1.main.reduction'))

    params = count_parameters(model)
    
    
    x=torch.randn(2,1,_cfg_require('GPN_V3_clean.ipynb.cell1.main.size_or_budget'),_cfg_require('GPN_V3_clean.ipynb.cell1.main.size_or_budget__2'))
    # 前向传播
    v, s = model(x)
    print(f"Embedding features (v) shape: {v.shape}")
    print(f"Precision features (s) shape: {s.shape}")
    print(f"Precision values are positive: {(s > 0).all().item()}")

    # 带中间特征输出
    v, s, features = model(x, return_intermediate=True)
    print(f"\nIntermediate features:")
    for name, feat in features.items():
        if isinstance(feat, torch.Tensor):
            print(f"  {name}: {feat.shape}")
    input_tensor = torch.randn(1, 1, _cfg_require('GPN_V3_clean.ipynb.cell1.main.size_or_budget__3'), _cfg_require('GPN_V3_clean.ipynb.cell1.main.size_or_budget__4'))
    
    flops, params = profile(model, inputs=(input_tensor,))
    flops, params = clever_format([flops, params], "%.3f")
    print(f"FLOPs: {flops}, Params: {params}")
    # 创建训练器
    meta_learner = GPNTrainer(model,device=device,
                              mlr=BLR,use_Mdistance=_cfg_require('GPN_V3_clean.ipynb.cell1.main.use_Mdistance'),use_multi=_cfg_require('GPN_V3_clean.ipynb.cell1.main.use_multi'),save_middle=_cfg_require('GPN_V3_clean.ipynb.cell1.main.save_middle'),
                              crop_ratio_h=_cfg_require('GPN_V3_clean.ipynb.cell1.main.crop_ratio_h'),crop_ratio_l=_cfg_require('GPN_V3_clean.ipynb.cell1.main.crop_ratio_l'),resize=_cfg_require('GPN_V3_clean.ipynb.cell1.main.resize'),
                              merge_threshold=_cfg_require('GPN_V3_clean.ipynb.cell1.main.merge_threshold'),merge_method=_cfg_require('GPN_V3_clean.ipynb.cell1.main.merge_method'))#('connected'连通分量'pairwise'成对合并 'hierarchical'层次聚类'dbscan'基于密度的聚类方法)
    print("开始训练...")
    meta_learner.train(train_loader=train_loader,test_loader=test_loader,epochs=EPOCHS,n_ways=N_WAY,save_path=SAVE_MODEL_PATH)
    print("训练完成！")
    
def inference():
    '\n    推理示例：加载已保存的模型进行预测\n    \n    !!!若报模型结构错误，检查：<configured>.引用是否为对应模型\n    '
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 创建模型
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,
                            train_ratio=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.train_ratio'),max_sample_count=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.max_sample_count'),
                            )
    model = GPN_Optimized(
                 reduction=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.reduction'),
                 )
    
    meta_learner = GPNTrainer(model,device=device,
                              mlr=BLR,use_Mdistance=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.use_Mdistance'),use_multi=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.use_multi'),save_middle=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.save_middle'),
                              crop_ratio_h=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.crop_ratio_h'),crop_ratio_l=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.crop_ratio_l'),resize=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.resize'),
                              merge_threshold=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.merge_threshold'),merge_method=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.merge_method'))
    meta_learner.load_model(LOAD_MODEL_PATH)
    
    profiler = FewShotDeploymentProfiler(
        trainer=meta_learner,
        device=device,
        input_size=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.input_size'),
    )

    report_gpn = profiler.generate_profile_report(
        test_dataset=test_dataset,
        n_way=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.n_way'),
        k_shot=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.k_shot'),
        q_query=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.q_query'),
        n_warmup=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.n_warmup'),
        n_trials=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.n_trials'),
        num_workers=_cfg_require('GPN_V3_clean.ipynb.cell1.inference.num_workers'),
    )
    PROFILE_DIR = Path(_cfg_require('GPN_V3_clean.ipynb.cell1.inference.path'))
    PROFILE_DIR.mkdir(parents=True, exist_ok=True)

    # 保存 JSON，保留完整字段
    with open(PROFILE_DIR / "report_gpn.json", "w", encoding="utf-8") as f:
        json.dump(report_gpn, f, indent=4, ensure_ascii=False)

    # 保存 CSV，方便论文表格整理
    pd.DataFrame([report_gpn]).to_csv(
        PROFILE_DIR / "report_gpn.csv",
        index=False,
        encoding="utf-8-sig"
    )
    # 评估
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.

    print("推理完成")
if __name__ == '__main__':
    # main()
    inference()
    
# % 时间计算：单任务构建+预测时间
# Historical configuration omitted; provide values through private configuration.
# % hierarchical：<configured>3ms
# % pairwise：<configured>5ms
# % dbscan：<configured>9ms
# % IMP：<configured>8ms
# %均采用warm-up策略去除第一个任务

In [ ]:
from private_config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
#主要变化：使用SiLU取代ReLU,优化模型架构，以此作为新模型迭代起点
from pathlib import Path
import json
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
from data_loader_clean import *
from GPN_V4_adaMulti_test import *
import higher
# from draw_pic import *
from thop import profile, clever_format
from para_and_indexes_collection_v1 import FewShotDeploymentProfiler
#约定：所有训练时使用类数<configured>抽<configured>，总样本<configured>个，每个任务中样本数支持集<configured>，查询集<configured>，
#     所有测试时使用类数<configured>抽<configured>-<configured>，总样本<configured>个，每个任务中样本数支持集<configured>,<configured>,<configured>，查询集<configured>，

# 参数设置
N_WAY = _cfg_require('GPN_V3_clean.ipynb.cell2.module.N_WAY')
K_SHOT = _cfg_require('GPN_V3_clean.ipynb.cell2.module.K_SHOT')
Q_QUERY = _cfg_require('GPN_V3_clean.ipynb.cell2.module.Q_QUERY')
NUM_TASKS = _cfg_require('GPN_V3_clean.ipynb.cell2.module.NUM_TASKS')
EPOCHS = _cfg_require('GPN_V3_clean.ipynb.cell2.module.EPOCHS')
BLR = _cfg_require('GPN_V3_clean.ipynb.cell2.module.BLR')
BATCH_SIZE =_cfg_require('GPN_V3_clean.ipynb.cell2.module.BATCH_SIZE')
SEED = _cfg_require('GPN_V3_clean.ipynb.cell2.module.SEED')#<configured>
SAVE_MODEL_DIR = Path(_cfg_require('GPN_V3_clean.ipynb.cell2.module.path'))
SAVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
# Historical configuration omitted; provide values through private configuration.
# Historical configuration omitted; provide values through private configuration.
SAVE_MODEL_PATH=SAVE_MODEL_DIR/_cfg_require('GPN_V3_clean.ipynb.cell2.module.path__2')
LOAD_MODEL_PATH=_cfg_require('GPN_V3_clean.ipynb.cell2.module.LOAD_MODEL_PATH')

# Historical configuration omitted; provide values through private configuration.
dataset_path = _cfg_require('GPN_V3_clean.ipynb.cell2.module.dataset_path')


def main():
    
    train_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=True,train_ratio=_cfg_require('GPN_V3_clean.ipynb.cell2.main.train_ratio'),max_sample_count=_cfg_require('GPN_V3_clean.ipynb.cell2.main.max_sample_count'),
                            )
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,train_ratio=_cfg_require('GPN_V3_clean.ipynb.cell2.main.train_ratio__2'),max_sample_count=_cfg_require('GPN_V3_clean.ipynb.cell2.main.max_sample_count__2'),
                            )

    meta_train_dataset = MetaDataset(train_dataset, 8,15,5,num_tasks=NUM_TASKS)
    meta_test_dataset = MetaDataset(test_dataset, N_WAY, K_SHOT, Q_QUERY,num_tasks=_cfg_require('GPN_V3_clean.ipynb.cell2.main.num_tasks'))
    
    # DataLoader
    train_loader = DataLoader(meta_train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=_cfg_require('GPN_V3_clean.ipynb.cell2.main.num_workers'))
    test_loader = DataLoader(meta_test_dataset, batch_size=_cfg_require('GPN_V3_clean.ipynb.cell2.main.batch_size'), shuffle=True)

    #创建模型
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = GPN_Optimized(
                 reduction=_cfg_require('GPN_V3_clean.ipynb.cell2.main.reduction'))

    params = count_parameters(model)
    
    
    x=torch.randn(2,1,_cfg_require('GPN_V3_clean.ipynb.cell2.main.size_or_budget'),_cfg_require('GPN_V3_clean.ipynb.cell2.main.size_or_budget__2'))
    # 前向传播
    v, s = model(x)
    print(f"Embedding features (v) shape: {v.shape}")
    print(f"Precision features (s) shape: {s.shape}")
    print(f"Precision values are positive: {(s > 0).all().item()}")

    # 带中间特征输出
    v, s, features = model(x, return_intermediate=True)
    print(f"\nIntermediate features:")
    for name, feat in features.items():
        if isinstance(feat, torch.Tensor):
            print(f"  {name}: {feat.shape}")
    input_tensor = torch.randn(1, 1, _cfg_require('GPN_V3_clean.ipynb.cell2.main.size_or_budget__3'), _cfg_require('GPN_V3_clean.ipynb.cell2.main.size_or_budget__4'))
    
    flops, params = profile(model, inputs=(input_tensor,))
    flops, params = clever_format([flops, params], "%.3f")
    print(f"FLOPs: {flops}, Params: {params}")
    # 创建训练器
    meta_learner = GPNTrainer(model,device=device,
                              mlr=BLR,use_Mdistance=_cfg_require('GPN_V3_clean.ipynb.cell2.main.use_Mdistance'),use_multi=_cfg_require('GPN_V3_clean.ipynb.cell2.main.use_multi'),save_middle=_cfg_require('GPN_V3_clean.ipynb.cell2.main.save_middle'),
                              crop_ratio_h=_cfg_require('GPN_V3_clean.ipynb.cell2.main.crop_ratio_h'),crop_ratio_l=_cfg_require('GPN_V3_clean.ipynb.cell2.main.crop_ratio_l'),resize=_cfg_require('GPN_V3_clean.ipynb.cell2.main.resize'),
                              merge_threshold=_cfg_require('GPN_V3_clean.ipynb.cell2.main.merge_threshold'),merge_method=_cfg_require('GPN_V3_clean.ipynb.cell2.main.merge_method'))#('connected'连通分量'pairwise'成对合并 'hierarchical'层次聚类'dbscan'基于密度的聚类方法)
    print("开始训练...")
    meta_learner.train(train_loader=train_loader,test_loader=test_loader,epochs=EPOCHS,n_ways=N_WAY,save_path=SAVE_MODEL_PATH)
    print("训练完成！")
    
def inference():
    '\n    推理示例：加载已保存的模型进行预测\n    \n    !!!若报模型结构错误，检查：<configured>.引用是否为对应模型\n    '
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 创建模型
    test_dataset=UAVDataset(data_dir_path=dataset_path,seed=SEED,is_train=False,
                            train_ratio=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.train_ratio'),max_sample_count=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.max_sample_count'),
                            )
    model = GPN_Optimized(
                 reduction=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.reduction'),
                 )
    
    meta_learner = GPNTrainer(model,device=device,
                              mlr=BLR,use_Mdistance=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.use_Mdistance'),use_multi=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.use_multi'),save_middle=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.save_middle'),
                              crop_ratio_h=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.crop_ratio_h'),crop_ratio_l=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.crop_ratio_l'),resize=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.resize'),
                              merge_threshold=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.merge_threshold'),merge_method=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.merge_method'))
    meta_learner.load_model(LOAD_MODEL_PATH)
    
    profiler = FewShotDeploymentProfiler(
        trainer=meta_learner,
        device=device,
        input_size=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.input_size'),
    )

    report_pn = profiler.generate_profile_report(
        test_dataset=test_dataset,
        n_way=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.n_way'),
        k_shot=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.k_shot'),
        q_query=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.q_query'),
        n_warmup=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.n_warmup'),
        n_trials=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.n_trials'),
        num_workers=_cfg_require('GPN_V3_clean.ipynb.cell2.inference.num_workers'),
    )
    PROFILE_DIR = Path(_cfg_require('GPN_V3_clean.ipynb.cell2.inference.path'))
    PROFILE_DIR.mkdir(parents=True, exist_ok=True)

    # 保存 JSON，保留完整字段
    with open(PROFILE_DIR / "report_pn.json", "w", encoding="utf-8") as f:
        json.dump(report_pn, f, indent=4, ensure_ascii=False)

    # 保存 CSV，方便论文表格整理
    pd.DataFrame([report_pn]).to_csv(
        PROFILE_DIR / "report_pn.csv",
        index=False,
        encoding="utf-8-sig"
    )
    # 评估
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.
    # Historical configuration omitted; provide values through private configuration.

    print("推理完成")
if __name__ == '__main__':
    # main()
    inference()
    
# % 时间计算：单任务构建+预测时间
# Historical configuration omitted; provide values through private configuration.
# % hierarchical：<configured>3ms
# % pairwise：<configured>5ms
# % dbscan：<configured>9ms
# % IMP：<configured>8ms
# %均采用warm-up策略去除第一个任务